In [3]:
import pandas as pd
df_train=pd.read_csv('data/train.csv')
df_test=pd.read_csv('data/test.csv')
df_train_merged = pd.read_csv('data/train_merged.csv')

df_train_merged.head()

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,SWI,MEDIUM,Mexico City Grand Prix,2023,0,6,1,6.0,12,83.921,-21.244,-10.320,0.084507,0.0,0.0
1,TRU,HARD,Italian Grand Prix,2024,0,24,2,17.0,15,83.845,-22.913,-33.696,0.311688,-9.0,1.0
2,TSU,MEDIUM,Monaco Grand Prix,2023,0,23,1,23.0,9,79.239,0.087,-12.078,0.302632,0.0,0.0
3,PEA,HARD,Italian Grand Prix,2022,1,50,2,33.0,11,87.076,-13.929,-31.804,0.694444,3.0,1.0
4,ANT,HARD,Monaco Grand Prix,2025,0,49,1,49.0,12,78.328,-0.516,-33.315,0.653333,0.0,0.0


In [4]:
from autogluon.tabular import TabularDataset, TabularPredictor

In [6]:
TARGET = 'PitNextLap'

print(df_train[TARGET].value_counts())
print(df_train_merged[TARGET].value_counts())

PitNextLap
0.0    351759
1.0     87381
Name: count, dtype: int64
PitNextLap
0.0    427273
1.0    113172
Name: count, dtype: int64


In [ ]:
# predictor = TabularPredictor(label=TARGET,eval_metric='roc_auc').fit(
#     train_data=df_train,
#     ag_args_fit={"num_gpus": 2},
#     time_limit=3600*9,
#     presets='best_quality',
#     verbosity=3,
#     num_stack_levels=0
# )

In [24]:
import warnings
warnings.filterwarnings(
    "ignore", category=FutureWarning, message=".*downcast.*"
)
_orig_fillna = pd.DataFrame.fillna
def _patched_fillna(self, *args, **kwargs):
    kwargs.pop("downcast", None)
    return _orig_fillna(self, *args, **kwargs)

In [ ]:
import os
import shutil
import pandas as pd
from autogluon.tabular import TabularPredictor

# ============================================================
# CONFIG
# ============================================================

MODEL_DIR = "./ag_models2"
TIME_LIMIT = 30 * 60  # 30 minutes (1800 seconds) per model family

models = {
    "GBM": [
        {},  # Generates LightGBM_BAG_L1
        {"extra_trees": True, "ag_args": {"name_suffix": "Large"}},  # Generates LightGBMLarge_BAG_L1
    ],
    "XGB": {},  # Generates XGBoost_BAG_L1
    "CAT": {},  # Generates CatBoost_BAG_L1
}

# ============================================================
# DELETE OLD MODELS
# ============================================================

# if os.path.exists(MODEL_DIR):
#     print(f"Deleting existing model directory: {MODEL_DIR}")
#     shutil.rmtree(MODEL_DIR)

# os.makedirs(MODEL_DIR, exist_ok=True)
print(f"Fresh model directory created: {MODEL_DIR}")

# ============================================================
# TRAIN ONE MODEL AT A TIME
# ============================================================

predictors = {}
results = []

for model_name, model_config in models.items():

    model_path = os.path.join(MODEL_DIR, model_name)

    print("\n" + "=" * 80)
    print(f"TRAINING: {model_name}")
    print(f"TIME LIMIT: 30 MINUTES")
    print(f"MODEL PATH: {model_path}")
    print("=" * 80)

    predictor = TabularPredictor(
        label=TARGET,
        eval_metric='roc_auc',
        path=model_path
    ).fit(
        train_data=df_train_merged,
        presets="best_quality",  # Enables bagging (_BAG_L1) and out-of-fold validation
        hyperparameters={model_name: model_config},  # Maps key to config dict properly
        ag_args_fit={
            'num_gpus': 1
        },
        time_limit=TIME_LIMIT,
        verbosity=3,
        num_stack_levels=0
    )

    predictors[model_name] = predictor

    # ========================================================
    # GET RESULT
    # ========================================================

    leaderboard = predictor.leaderboard(silent=True)
    best_row = leaderboard.iloc[0]

    results.append({
        'Model Family': model_name,
        'Best AutoGluon Model': best_row['model'],
        'Validation ROC-AUC': best_row['score_val'],
        'Fit Time (sec)': best_row['fit_time'],
        'Predict Time (sec)': best_row['pred_time_val']
    })

    print(f"\n{model_name} completed.")
    print(f"Validation ROC-AUC: {best_row['score_val']:.6f}")

# ============================================================
# FINAL COMPARISON
# ============================================================

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    'Validation ROC-AUC',
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 80)
print("FINAL MODEL COMPARISON")
print("=" * 80)

display(results_df)

Verbosity: 3 (Detailed Logging)


=================== System Info ===================
AutoGluon Version:  1.6.1
Python Version:     3.13.13
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.13.0+cu126
CUDA Version:       12.6
GPU Memory:         GPU 0: 15.93/15.93 GB
Total GPU Memory:   Free: 15.93 GB, Allocated: 0.00 GB, Total: 15.93 GB
GPU Count:          1
Memory Avail:       3.16 GB / 15.06 GB (21.0%)
Disk Space Avail:   712.37 GB / 930.47 GB (76.6%)
Presets specified: ['best_quality']
============ fit kwarg info ============
User Specified kwargs:
{'ag_args_fit': {'num_gpus': 1},
 'auto_stack': True,
 'num_stack_levels': 0,
 'verbosity': 3}
Full kwargs:
{'_experimental_dynamic_hyperparameters': False,
 '_feature_generator_kwargs': None,
 '_save_bag_folds': None,
 'adapt_num_bag_folds_to_n_classes': False,
 'ag_args': None,
 'ag_args_ensemble': None,
 'ag_args_fit': {'num_gpus': 1},
 'auto_stack': True,
 'aux_kwargs': None,
 'calibrate'

Deleting existing model directory: ./ag_models2
Fresh model directory created: ./ag_models2

TRAINING: GBM
TIME LIMIT: 30 MINUTES
MODEL PATH: ./ag_models2\GBM


Beginning AutoGluon training ... Time limit = 1800s
AutoGluon will save models to "c:\Darshak\Projects\Hackathon\ag_models2\GBM"
Train Data Rows:    540445
Train Data Columns: 14
Label Column:       PitNextLap
AutoGluon infers your prediction problem is: 'binary' (because only two unique label-values observed).
	2 unique label values:  [np.float64(0.0), np.float64(1.0)]
	If 'binary' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify problem_type as one of: ['binary', 'multiclass', 'regression', 'quantile'])
Problem Type:       binary
Preprocessing data...
Selected class <--> label mapping:  class 1 = 1, class 0 = 0
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    3216.31 MB
	Train Data (Original)  Memory Usage: 136.13 MB (4.2% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to

[50]	valid_set's binary_logloss: 0.281445
[100]	valid_set's binary_logloss: 0.260036
[150]	valid_set's binary_logloss: 0.25199
[200]	valid_set's binary_logloss: 0.247187
[250]	valid_set's binary_logloss: 0.243627
[300]	valid_set's binary_logloss: 0.240783
[350]	valid_set's binary_logloss: 0.23862
[400]	valid_set's binary_logloss: 0.236582
[450]	valid_set's binary_logloss: 0.234896
[500]	valid_set's binary_logloss: 0.233701
[550]	valid_set's binary_logloss: 0.232697
[600]	valid_set's binary_logloss: 0.231964
[650]	valid_set's binary_logloss: 0.231146
[700]	valid_set's binary_logloss: 0.230417
[750]	valid_set's binary_logloss: 0.22974
[800]	valid_set's binary_logloss: 0.229188
[850]	valid_set's binary_logloss: 0.228568
[900]	valid_set's binary_logloss: 0.228074
[950]	valid_set's binary_logloss: 0.227555
[1000]	valid_set's binary_logloss: 0.227149
[1050]	valid_set's binary_logloss: 0.226735
[1100]	valid_set's binary_logloss: 0.226487
[1150]	valid_set's binary_logloss: 0.226146
[1200]	vali

...

	Fit constraints were validated upstream on the full training data, skipping...
	Training S1F8 with GPU, note that this may negatively impact model quality compared to CPU training.
	Catboost model hyperparameters: {'iterations': 10000, 'learning_rate': 0.05, 'allow_writing_files': False, 'eval_metric': 'Logloss', 'random_seed': 0, 'thread_count': 16, 'task_type': 'GPU'}


0:	learn: 0.6645892	test: 0.6644791	best: 0.6644791 (0)	total: 5.69ms	remaining: 5.69ms
1:	learn: 0.6387453	test: 0.6385256	best: 0.6385256 (1)	total: 10.2ms	remaining: 0us
bestTest = 0.6385255533
bestIteration = 1
0:	learn: 0.6400220	test: 0.6398224	best: 0.6398224 (0)	total: 18.8ms	remaining: 1m 6s
20:	learn: 0.3348168	test: 0.3329328	best: 0.3329328 (20)	total: 411ms	remaining: 1m 9s
40:	learn: 0.3030511	test: 0.3002934	best: 0.3002934 (40)	total: 803ms	remaining: 1m 9s
60:	learn: 0.2899778	test: 0.2866054	best: 0.2866054 (60)	total: 1.18s	remaining: 1m 7s
80:	learn: 0.2818726	test: 0.2781329	best: 0.2781329 (80)	total: 1.56s	remaining: 1m 7s
100:	learn: 0.2770052	test: 0.2733906	best: 0.2733906 (100)	total: 1.94s	remaining: 1m 6s
120:	learn: 0.2721739	test: 0.2684802	best: 0.2684802 (120)	total: 2.3s	remaining: 1m 5s
140:	learn: 0.2672213	test: 0.2635353	best: 0.2635353 (140)	total: 2.68s	remaining: 1m 5s
160:	learn: 0.2635022	test: 0.2598519	best: 0.2598519 (160)	total: 3.04s	rema

Saving c:\Darshak\Projects\Hackathon\ag_models2\CAT\models\CatBoost_BAG_L1\utils\oof.pkl
Saving c:\Darshak\Projects\Hackathon\ag_models2\CAT\models\CatBoost_BAG_L1\model.pkl
	0.9508	 = Validation score   (roc_auc)
	204.68s	 = Training   runtime
	0.83s	 = Validation runtime
	81567.8	 = Inference  throughput (rows/s | 67556 batch size)
Saving c:\Darshak\Projects\Hackathon\ag_models2\CAT\models\trainer.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\CAT\models\CatBoost_BAG_L1\utils\oof.pkl
Model configs that will be trained (in order):
	WeightedEnsemble_L2: 	{'ag_args': {'problem_types': ['binary', 'multiclass', 'regression', 'quantile', 'softclass'], 'valid_base': False, 'name_bag_suffix': '', 'model_type': <class 'autogluon.core.models.greedy_ensemble.greedy_weighted_ensemble_model.GreedyWeightedEnsembleModel'>, 'priority': 0}, 'ag_args_ensemble': {'save_bag_folds': True}}
Fitting model: WeightedEnsemble_L2 ... Training model for up to 360.00s of the 1590.90s of remaining time.
	F


CAT completed.
Validation ROC-AUC: 0.950763

FINAL MODEL COMPARISON


,Model Family,Best AutoGluon Model,Validation ROC-AUC,Fit Time (sec),Predict Time (sec)
0,XGB,XGBoost_BAG_L1,0.958240,602.314523,4.416461
1,GBM,WeightedEnsemble_L2,0.953617,256.293446,19.325813
2,CAT,CatBoost_BAG_L1,0.950763,204.676705,0.828219


In [29]:
import os
import pandas as pd
from autogluon.tabular import TabularPredictor

# ============================================================
# CONFIG
# ============================================================

MODEL_DIR = "./ag_models2"

all_leaderboards = []

# ============================================================
# SCAN & LOAD ALL TRAINED PREDICTORS
# ============================================================

if os.path.exists(MODEL_DIR):
    # Check both subdirectories and root directory for saved predictors
    folder_candidates = [MODEL_DIR] + [
        os.path.join(MODEL_DIR, d) 
        for d in os.listdir(MODEL_DIR) 
        if os.path.isdir(os.path.join(MODEL_DIR, d))
    ]

    for folder_path in folder_candidates:
        # Check if directory contains a valid predictor file
        if os.path.exists(os.path.join(folder_path, "predictor.pkl")):
            try:
                folder_name = os.path.basename(folder_path)
                print(f"Loading predictor from: {folder_path}")
                
                # Load predictor from disk
                predictor = TabularPredictor.load(folder_path)
                
                # Fetch leaderboard
                lb = predictor.leaderboard(silent=True)
                lb.insert(0, "folder_name", folder_name)
                
                all_leaderboards.append(lb)
            except Exception as e:
                print(f"Failed to load predictor from {folder_path}: {e}")

# ============================================================
# MERGE AND PRESENT COMBINED LEADERBOARD
# ============================================================

if all_leaderboards:
    combined_leaderboard = pd.concat(all_leaderboards, ignore_index=True)

    # Sort all trained models by validation ROC-AUC score descending
    combined_leaderboard = combined_leaderboard.sort_values(
        by="score_val", ascending=False
    ).reset_index(drop=True)

    print("\n" + "=" * 80)
    print("COMBINED LEADERBOARD OF ALL LOADED MODELS")
    print("=" * 80)

    display(combined_leaderboard)
else:
    print(f"No valid AutoGluon predictors (`predictor.pkl`) found under '{MODEL_DIR}'.")

Loading predictor from: ./ag_models2\CAT


Loading: c:\Darshak\Projects\Hackathon\ag_models2\CAT\predictor.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\CAT\learner.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\CAT\models\trainer.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\GBM\predictor.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\GBM\learner.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\GBM\models\trainer.pkl


Loading predictor from: ./ag_models2\GBM
Loading predictor from: ./ag_models2\RF


Loading: c:\Darshak\Projects\Hackathon\ag_models2\RF\predictor.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\RF\learner.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\RF\models\trainer.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\predictor.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\learner.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\trainer.pkl


Loading predictor from: ./ag_models2\XGB

COMBINED LEADERBOARD OF ALL LOADED MODELS


,folder_name,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,XGB,WeightedEnsemble_L2,0.958240,roc_auc,4.397609,601.833442,0.053591,0.057095,2.0,1.0,2.0
1,XGB,XGBoost_BAG_L1,0.958240,roc_auc,4.344018,601.776347,4.344018,601.776347,1.0,1.0,1.0
2,GBM,WeightedEnsemble_L2,0.953617,roc_auc,18.501622,255.991845,0.060542,2.829900,2.0,1.0,3.0
3,GBM,LightGBMLarge_BAG_L1,0.952897,roc_auc,13.476475,172.914947,13.476475,172.914947,1.0,1.0,2.0
4,GBM,LightGBM_BAG_L1,0.951028,roc_auc,4.964605,80.246999,4.964605,80.246999,1.0,1.0,1.0
5,CAT,WeightedEnsemble_L2,0.950199,roc_auc,0.780571,179.753519,0.057653,0.067708,2.0,1.0,2.0
6,CAT,CatBoost_BAG_L1,0.950199,roc_auc,0.722918,179.685812,0.722918,179.685812,1.0,1.0,1.0


In [75]:
predictor = TabularPredictor.load("ag_models2/XGB")
predictor.info()

Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\predictor.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\learner.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\trainer.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_BAG_L1\S1F1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_BAG_L1\S1F2\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_BAG_L1\S1F3\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_BAG_L1\S1F4\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_BAG_L1\S1F5\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_BAG_L1\S1F6\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_BAG_L1\S1F7\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_B

{'path': 'c:\\Darshak\\Projects\\Hackathon\\ag_models2\\XGB',
 'label': 'PitNextLap',
 'random_state': 0,
 'version': '1.6.1',
 'features': ['Driver',
  'Compound',
  'Race',
  'Year',
  'PitStop',
  'LapNumber',
  'Stint',
  'TyreLife',
  'Position',
  'LapTime (s)',
  'LapTime_Delta',
  'Cumulative_Degradation',
  'RaceProgress',
  'Position_Change'],
 'feature_metadata_in': <autogluon.common.features.feature_metadata.FeatureMetadata at 0x18a176d61b0>,
 'time_fit_preprocessing': 0.8181874752044678,
 'time_fit_training': 607.6769812107086,
 'time_fit_total': 608.4951686859131,
 'time_limit': 1800,
 'time_train_start': 1786370652.9010363,
 'num_rows_train': 540445,
 'num_cols_train': 14,
 'num_rows_val': None,
 'num_rows_test': None,
 'num_classes': 2,
 'problem_type': 'binary',
 'eval_metric': 'roc_auc',
 'best_model': 'WeightedEnsemble_L2',
 'best_model_score_val': np.float64(0.9582403341215795),
 'best_model_stack_level': 2,
 'num_models_trained': 2,
 'num_bag_folds': 8,
 'max_stack

In [76]:
df=predictor.predict_proba(df_test)
df.head()

Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\WeightedEnsemble_L2\model.pkl


,0,1
0,0.995741,0.004259
1,0.995626,0.004374
2,0.996494,0.003506
3,0.800449,0.199551
4,0.039684,0.960316


In [77]:
# df.to_csv("With_driver_feature_submission.csv")
df_sample_out=pd.read_csv('data/sample_submission.csv')
df_sample_out.head()

,id,PitNextLap
0,439140,0
1,439141,0
2,439142,0
3,439143,0
4,439144,0


In [78]:
df_sample_out['PitNextLap']=df[1]

In [79]:
df_sample_out.head()

,id,PitNextLap
0,439140,0.004259
1,439141,0.004374
2,439142,0.003506
3,439143,0.199551
4,439144,0.960316


In [80]:
df_sample_out.to_csv("My_output/With_driver_submission.csv")

Loading: c:\Darshak\Projects\Hackathon\ag_models2\CAT\models\CatBoost_BAG_L1\model.pkl


Loading: c:\Darshak\Projects\Hackathon\ag_models2\CAT\models\WeightedEnsemble_L2\model.pkl


      PREDICTION ANALYSIS REPORT        
Total Records                      : 10000
Correct Matches                    : 9126
Mismatches                         : 874
Accuracy (%)                       : 91.26
True Positives (Actual 1, Pred 1)  : 1527
True Negatives (Actual 0, Pred 0)  : 7599
False Positives (Actual 0, Pred 1) : 401
False Negatives (Actual 1, Pred 0) : 473
Loaded: sample_part_1_10000k.csv -> Shape: (10000, 15)


In [81]:
from multi_sampling_test_predictor import process_and_evaluate_all_csvs
data_dict = process_and_evaluate_all_csvs(predictor,folder_path="Sampling_data_to_test",drop_cols=[])

Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_BAG_L1\model.pkl



----------------------------------------
 Processing: sample_part_1_10000k.csv
----------------------------------------
ground_truth
0    8000
1    2000
Name: count, dtype: int64


Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\WeightedEnsemble_L2\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_BAG_L1\model.pkl


--- Per-File Analysis [sample_part_1_10000k.csv] ---
      PREDICTION ANALYSIS REPORT        
Total Records                      : 10000
Correct Matches                    : 9453
Mismatches                         : 547
Accuracy (%)                       : 94.53
True Positives (Actual 1, Pred 1)  : 1716
True Negatives (Actual 0, Pred 0)  : 7737
False Positives (Actual 0, Pred 1) : 263
False Negatives (Actual 1, Pred 0) : 284

----------------------------------------
 Processing: sample_part_2_10000k.csv
----------------------------------------
ground_truth
0    8500
1    1500
Name: count, dtype: int64


Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\WeightedEnsemble_L2\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_BAG_L1\model.pkl


...

--- Per-File Analysis [sample_part_3_10000k.csv] ---
      PREDICTION ANALYSIS REPORT        
Total Records                      : 10000
Correct Matches                    : 9424
Mismatches                         : 576
Accuracy (%)                       : 94.24
True Positives (Actual 1, Pred 1)  : 2161
True Negatives (Actual 0, Pred 0)  : 7263
False Positives (Actual 0, Pred 1) : 237
False Negatives (Actual 1, Pred 0) : 339

----------------------------------------
 Processing: sample_part_4_10000k.csv
----------------------------------------
ground_truth
0    7000
1    3000
Name: count, dtype: int64


Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\WeightedEnsemble_L2\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_BAG_L1\model.pkl


--- Per-File Analysis [sample_part_4_10000k.csv] ---
      PREDICTION ANALYSIS REPORT        
Total Records                      : 10000
Correct Matches                    : 9421
Mismatches                         : 579
Accuracy (%)                       : 94.21
True Positives (Actual 1, Pred 1)  : 2618
True Negatives (Actual 0, Pred 0)  : 6803
False Positives (Actual 0, Pred 1) : 197
False Negatives (Actual 1, Pred 0) : 382

----------------------------------------
 Processing: sample_part_5_10000k.csv
----------------------------------------
ground_truth
0    9000
1    1000
Name: count, dtype: int64


Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\WeightedEnsemble_L2\model.pkl


--- Per-File Analysis [sample_part_5_10000k.csv] ---
      PREDICTION ANALYSIS REPORT        
Total Records                      : 10000
Correct Matches                    : 9575
Mismatches                         : 425
Accuracy (%)                       : 95.75
True Positives (Actual 1, Pred 1)  : 851
True Negatives (Actual 0, Pred 0)  : 8724
False Positives (Actual 0, Pred 1) : 276
False Negatives (Actual 1, Pred 0) : 149

      OVERALL CUMULATIVE ANALYSIS REPORT        
      PREDICTION ANALYSIS REPORT        
Total Records                      : 50000
Correct Matches                    : 47409
Mismatches                         : 2591
Accuracy (%)                       : 94.82
True Positives (Actual 1, Pred 1)  : 8666
True Negatives (Actual 0, Pred 0)  : 38743
False Positives (Actual 0, Pred 1) : 1257
False Negatives (Actual 1, Pred 0) : 1334
